# 03. Preprocesamiento de Datos

## Objetivo

El objetivo de esta etapa fue preparar el dataset para el entrenamiento de modelos de Machine Learning, resolviendo problemas de calidad de datos y transformando las variables a un formato adecuado para modelado.

Durante esta fase se realizaron:

- análisis de tipos de variables,
- evaluación de valores faltantes,
- eliminación de variables con missing extremo,
- creación de señales derivadas de missing values,
- imputación de datos,
- codificación de variables categóricas,
- y división temporal de entrenamiento y validación.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

train = pd.read_parquet(
    "data/processed/train_merged.parquet"
)

: 

In [11]:
# ==========================================
# VARIABLE INVENTORY
# ==========================================

numeric_cols = train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_cols = train.select_dtypes(
    include=["object"]
).columns.tolist()

print("="*50)
print("NUMERIC VARIABLES")
print("="*50)
print(len(numeric_cols))

print("\nExamples:")
print(numeric_cols[:20])

print("\n" + "="*50)
print("CATEGORICAL VARIABLES")
print("="*50)
print(len(categorical_cols))

print("\nExamples:")
print(categorical_cols[:20])

NUMERIC VARIABLES
403

Examples:
['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2', 'dist1', 'dist2', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8']

CATEGORICAL VARIABLES
31

Examples:
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28']


/tmp/ipykernel_1509/4288217037.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = train.select_dtypes(


In [12]:
# ==========================================
# MISSING PROFILE
# ==========================================

missing = (
    train.isna()
    .mean()
    .sort_values(ascending=False)
    * 100
)

missing_df = pd.DataFrame({
    "missing_pct": missing
})

display(
    missing_df.head(40)
)

,missing_pct
id_24,99.196159
id_25,99.130965
id_07,99.127070
id_08,99.127070
id_21,99.126393
id_26,99.125715
id_22,99.124699
id_27,99.124699
id_23,99.124699
dist2,93.628374


In [13]:
# ==========================================
# VARIABLES WITH >95% MISSING
# ==========================================

high_missing = (
    train.isna()
    .mean()
)

high_missing = high_missing[
    high_missing > 0.95
]

print(
    "Variables >95% missing:",
    len(high_missing)
)

display(
    high_missing.sort_values(
        ascending=False
    )
)

Variables >95% missing: 9


id_24    0.991962
id_25    0.991310
id_07    0.991271
id_08    0.991271
id_21    0.991264
id_26    0.991257
id_22    0.991247
id_23    0.991247
id_27    0.991247
dtype: float64

In [14]:
# ==========================================
# HIGH MISSING SIGNAL CHECK
# ==========================================

high_missing_cols = [
    "id_24",
    "id_25",
    "id_07",
    "id_08",
    "id_21",
    "id_22",
    "id_23",
    "id_26",
    "id_27"
]

results = []

for col in high_missing_cols:

    fraud_missing = (
        train.loc[
            train["isFraud"] == 1,
            col
        ]
        .isna()
        .mean()
        * 100
    )

    nonfraud_missing = (
        train.loc[
            train["isFraud"] == 0,
            col
        ]
        .isna()
        .mean()
        * 100
    )

    diff = abs(
        fraud_missing -
        nonfraud_missing
    )

    results.append([
        col,
        round(nonfraud_missing, 2),
        round(fraud_missing, 2),
        round(diff, 2)
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "variable",
        "non_fraud_missing_pct",
        "fraud_missing_pct",
        "difference"
    ]
)

display(
    results_df.sort_values(
        by="difference",
        ascending=False
    )
)

,variable,non_fraud_missing_pct,fraud_missing_pct,difference
2,id_07,99.17,97.94,1.23
7,id_26,99.17,97.94,1.23
3,id_08,99.17,97.94,1.23
4,id_21,99.17,97.94,1.23
5,id_22,99.17,97.94,1.23
8,id_27,99.17,97.94,1.23
6,id_23,99.17,97.94,1.23
1,id_25,99.17,97.98,1.19
0,id_24,99.24,98.05,1.18


In [15]:
# ==========================================
# DROP HIGH MISSING VARIABLES
# ==========================================

cols_drop = [
    "id_24",
    "id_25",
    "id_07",
    "id_08",
    "id_21",
    "id_22",
    "id_23",
    "id_26",
    "id_27"
]

train = train.drop(
    columns=cols_drop
)

print(train.shape)

(590540, 425)


In [16]:
# Recalcular listas después de eliminar columnas

numeric_cols = train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_cols = train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print("Numeric:", len(numeric_cols))
print("Categorical:", len(categorical_cols))

Numeric: 396
Categorical: 29


In [17]:
# ==========================================
# CATEGORICAL CARDINALITY
# ==========================================

cardinality = []

for col in categorical_cols:
    n_unique = train[col].nunique(dropna=True)
    cardinality.append([col, n_unique])

cardinality_df = pd.DataFrame(
    cardinality,
    columns=["variable", "n_unique"]
)

display(
    cardinality_df.sort_values(
        by="n_unique",
        ascending=False
    )
)

,variable,n_unique
28,DeviceInfo,1786
21,id_33,260
20,id_31,130
19,id_30,75
4,R_emaildomain,60
3,P_emaildomain,59
0,ProductCD,5
2,card6,4
22,id_34,4
1,card4,4


In [18]:
# ==========================================
# DEFINE CATEGORICAL STRATEGY
# ==========================================

high_cardinality_cat = [
    "DeviceInfo",
    "id_33",
    "id_31",
    "id_30",
    "R_emaildomain",
    "P_emaildomain"
]

low_cardinality_cat = [
    col for col in categorical_cols
    if col not in high_cardinality_cat
]

print("High cardinality categorical:")
print(high_cardinality_cat)

print("\nLow cardinality categorical:")
print(low_cardinality_cat)

print("\nCounts:")
print("High:", len(high_cardinality_cat))
print("Low:", len(low_cardinality_cat))

High cardinality categorical:
['DeviceInfo', 'id_33', 'id_31', 'id_30', 'R_emaildomain', 'P_emaildomain']

Low cardinality categorical:
['ProductCD', 'card4', 'card6', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType']

Counts:
High: 6
Low: 23


In [19]:
# ==========================================
# TEMPORAL TRAIN / VALIDATION SPLIT
# ==========================================

train = train.sort_values(
    "TransactionDT"
).reset_index(drop=True)

split_index = int(
    len(train) * 0.8
)

train_df = train.iloc[
    :split_index
].copy()

valid_df = train.iloc[
    split_index:
].copy()

print("Train shape:")
print(train_df.shape)

print("\nValidation shape:")
print(valid_df.shape)

print("\nFraud rate train:")
print(
    train_df["isFraud"]
    .mean() * 100
)

print("\nFraud rate validation:")
print(
    valid_df["isFraud"]
    .mean() * 100
)

Train shape:
(472432, 425)

Validation shape:
(118108, 425)

Fraud rate train:
3.5135215226741625

Fraud rate validation:
3.4409184813899145


In [20]:
# ==========================================
# CREATE MISSING INDICATORS
# ==========================================

missing_signal_cols = [
    "DeviceType",
    "DeviceInfo",
    "id_31",
    "dist1",
    "dist2"
]

for col in missing_signal_cols:

    train_df[f"{col}_missing"] = (
        train_df[col]
        .isna()
        .astype(int)
    )

    valid_df[f"{col}_missing"] = (
        valid_df[col]
        .isna()
        .astype(int)
    )

print("Done")

Done


In [21]:
# ==========================================
# IMPUTE LOW CARDINALITY CATEGORICAL
# ==========================================

for col in low_cardinality_cat:

    train_df[col] = train_df[
        col
    ].fillna("Missing")

    valid_df[col] = valid_df[
        col
    ].fillna("Missing")

print("Done")

Done


In [22]:
# ==========================================
# FREQUENCY ENCODING
# ==========================================

for col in high_cardinality_cat:

    freq_encoding = (
        train_df[col]
        .value_counts(
            normalize=True
        )
    )

    train_df[col] = (
        train_df[col]
        .map(freq_encoding)
    )

    valid_df[col] = (
        valid_df[col]
        .map(freq_encoding)
    )

print("Done")

Done


In [ ]:
# ==========================================
# NUMERIC IMPUTATION (MEDIAN) - FIXED
# ==========================================

numeric_features = train_df.select_dtypes(
    include=["number"]
).columns.tolist()

numeric_features = [
    col for col in numeric_features
    if col not in ["isFraud", "TransactionID"]
]

median_values = train_df[numeric_features].median(
    numeric_only=True
)

train_df[numeric_features] = train_df[
    numeric_features
].fillna(median_values)

valid_df[numeric_features] = valid_df[
    numeric_features
].fillna(median_values)

print("Done")
print("Numeric features:", len(numeric_features))

Done
Numeric features: 405


In [25]:
# ==========================================
# CHECK REMAINING MISSING
# ==========================================

missing_after = (
    train_df.isna()
    .mean()
    .sort_values(ascending=False)
)

remaining_missing = missing_after[
    missing_after > 0
]

print(
    "Columns with missing:",
    len(remaining_missing)
)

display(
    remaining_missing.head(30)
)

Columns with missing: 0


Series([], dtype: float64)

In [26]:
# ==========================================
# SAVE PREPROCESSED CHECKPOINT
# ==========================================

train_df.to_parquet(
    "data/processed/train_preprocessed_temporal.parquet",
    index=False
)

valid_df.to_parquet(
    "data/processed/valid_preprocessed_temporal.parquet",
    index=False
)

print("Saved")

Saved


In [27]:
import gc

gc.collect()

58

In [28]:
# ==========================================
# ONE-HOT ENCODING LOW CARDINALITY
# ==========================================

train_df_encoded = pd.get_dummies(
    train_df,
    columns=low_cardinality_cat,
    drop_first=False
)

valid_df_encoded = pd.get_dummies(
    valid_df,
    columns=low_cardinality_cat,
    drop_first=False
)

# Alinear columnas para que train y valid tengan exactamente las mismas
train_df_encoded, valid_df_encoded = train_df_encoded.align(
    valid_df_encoded,
    join="left",
    axis=1,
    fill_value=0
)

print("Train encoded:", train_df_encoded.shape)
print("Valid encoded:", valid_df_encoded.shape)

Train encoded: (472432, 486)
Valid encoded: (118108, 486)


In [29]:
print("Missing train:", train_df_encoded.isna().sum().sum())
print("Missing valid:", valid_df_encoded.isna().sum().sum())

print("Columnas iguales:")
print(list(train_df_encoded.columns) == list(valid_df_encoded.columns))

Missing train: 0
Missing valid: 0
Columnas iguales:
True


In [30]:
train_df_encoded.to_parquet(
    "data/processed/train_ready_modeling.parquet",
    index=False
)

valid_df_encoded.to_parquet(
    "data/processed/valid_ready_modeling.parquet",
    index=False
)

print("Saved")

Saved
